# Evidence-First Pipeline — RCA ablation (summary)

Summarized results of the **15-incident x 3-rep** ablation on the RCA agent,
scored by an independent gpt-5 judge. Orange = without pipeline (raw collect-all),
blue = with pipeline (distilled). Numbers are baked in below — no data files needed.

**Headline:** localization accuracy is a tie (89.2% vs 89.2%);
the pipeline matches accuracy at ~52x fewer tokens and comparable latency.

In [ ]:
import matplotlib.pyplot as plt

# --- summarized results (15 incidents x 3 reps, gpt-5 judge) ---
buckets = ['1 fault', '2 faults', '4 faults']
modes   = ['raw', 'distilled']

# localization accuracy (%), micro-averaged over root causes
acc = {'raw': [100.0, 100.0, 72.2], 'distilled': [100.0, 97.6, 75.0]}
# diagnosis latency (s), mean
lat = {'raw': [51.7, 50.1, 63.6], 'distilled': [47.0, 50.0, 65.3]}
# prompt context (tokens), mean
tok = {'raw': [38332, 39168, 42238], 'distilled': [726, 815, 988]}
# overall, across all fault-judgments
overall   = {'raw': 89.2, 'distilled': 89.2}
precision = {'raw': 100.0, 'distilled': 100.0}

COL = {'raw':'#f0871e', 'distilled':'#2f7dc4'}
LBL = {'raw':'Without pipeline (raw)', 'distilled':'With pipeline (distilled)'}

def kfmt(v):  # compact token label: 726 -> '726', 38332 -> '38k'
    return f'{v/1000:.0f}k' if v >= 1000 else f'{v:.0f}'

In [ ]:
# --- three panels: accuracy / latency / tokens by difficulty (all annotated) ---
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))

def kfmt(v):  # compact token label (self-contained)
    return f'{v/1000:.0f}k' if v >= 1000 else f'{v:.0f}'

# panel 0: accuracy
for m in modes:
    ax[0].plot(buckets, acc[m], marker='o', lw=2.4, color=COL[m], label=LBL[m])
    for xi, yi in zip(buckets, acc[m]):
        ax[0].annotate(f'{yi:.0f}%', (xi, yi), textcoords='offset points', xytext=(0, 8),
                       ha='center', color=COL[m], fontweight='bold')
ax[0].set_title('Localization accuracy by difficulty'); ax[0].set_ylim(0, 108)
ax[0].set_ylabel('% root causes found (gpt-5 judge)'); ax[0].legend(loc='lower left'); ax[0].grid(alpha=.3)

# panel 1: latency
for m in modes:
    ax[1].plot(buckets, lat[m], marker='o', lw=2.4, color=COL[m], label=LBL[m])
    for xi, yi in zip(buckets, lat[m]):
        ax[1].annotate(f'{yi:.0f}s', (xi, yi), textcoords='offset points', xytext=(0, 8),
                       ha='center', color=COL[m], fontweight='bold')
ax[1].set_title('Diagnosis latency by difficulty'); ax[1].set_ylabel('seconds (mean)')
ax[1].set_ylim(0, max(max(lat['raw']), max(lat['distilled'])) * 1.18); ax[1].grid(alpha=.3); ax[1].legend()

# panel 2: tokens (log)
for m in modes:
    ax[2].plot(buckets, tok[m], marker='o', lw=2.4, color=COL[m], label=LBL[m])
    for xi, yi in zip(buckets, tok[m]):
        va = 'bottom' if m == 'raw' else 'top'
        dy = 9 if m == 'raw' else -11
        ax[2].annotate(kfmt(yi), (xi, yi), textcoords='offset points', xytext=(0, dy),
                       ha='center', va=va, color=COL[m], fontweight='bold')
ax[2].set_yscale('log'); ax[2].set_ylim(300, 100000); ax[2].set_title('Prompt context by difficulty')
ax[2].set_ylabel('tokens (log scale)'); ax[2].grid(alpha=.3, which='both'); ax[2].legend()

plt.suptitle(f'RCA ablation  —  overall accuracy: raw {overall["raw"]}%  vs  distilled {overall["distilled"]}%   '
             f'(~{tok["raw"][0]//tok["distilled"][0]}x fewer tokens)', fontsize=12)
plt.tight_layout(); plt.savefig('ablation_by_difficulty.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# --- overall summary as grouped bars ---
import numpy as np
metrics = ['Accuracy %', 'Precision %', 'Latency (s)', 'Tokens (k)']
raw_v = [overall['raw'], precision['raw'], sum(lat['raw'])/3, sum(tok['raw'])/3/1000]
dst_v = [overall['distilled'], precision['distilled'], sum(lat['distilled'])/3, sum(tok['distilled'])/3/1000]
xi = np.arange(len(metrics)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 4.3))
b1 = ax.bar(xi-w/2, raw_v, w, color=COL['raw'], label=LBL['raw'])
b2 = ax.bar(xi+w/2, dst_v, w, color=COL['distilled'], label=LBL['distilled'])
ax.set_xticks(xi); ax.set_xticklabels(metrics)
ax.bar_label(b1, fmt='%.1f', fontsize=8); ax.bar_label(b2, fmt='%.1f', fontsize=8)
ax.legend(); ax.grid(alpha=.3, axis='y'); ax.set_title('Overall (mean over 15 incidents x 3 reps)')
plt.tight_layout(); plt.savefig('ablation_overall.png', dpi=150, bbox_inches='tight'); plt.show()